# Keyword Spotting (Toy Demo)

A lightweight demo showing how to prepare mel features and train a small CNN on a tiny split of the Speech Commands dataset. Intended for Colab (GPU recommended).
This notebook uses `tensorflow` and `tensorflow_datasets` and keeps the dataset tiny for quick runs.

In [1]:
!pip install tensorflow tensorflow-datasets librosa -q

In [7]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import librosa
import matplotlib.pyplot as plt

print('TF version:', tf.__version__)

TF version: 2.19.0


## Load tiny subset of Speech Commands
We load a small fraction to keep runtime low. Replace split ranges to use more data.

In [9]:
# Try the small mini_speech_commands dataset; fall back to speech_commands if unavailable on this TFDS version
try:
    dataset_name = 'mini_speech_commands'
    train_split = 'train[:80%]'
    test_split = 'train[80%:]'
    (raw_train, raw_test), ds_info = tfds.load(
        dataset_name,
        split=[train_split, test_split],
        as_supervised=True,
        shuffle_files=False,
        with_info=True,
    )
    used_dataset = dataset_name
except tfds.core.DatasetNotFoundError:
    # Fallback: use a tiny slice of the original speech_commands to keep downloads light
    dataset_name = 'speech_commands'
    train_split = 'train[:1%]'
    test_split = 'test[:1%]'
    (raw_train, raw_test), ds_info = tfds.load(
        dataset_name,
        split=[train_split, test_split],
        as_supervised=True,
        shuffle_files=False,
        with_info=True,
    )
    used_dataset = dataset_name

label_names = ds_info.features['label'].names
num_classes = len(label_names)

# Shuffle only in-memory buffer to keep startup fast
ds_train = raw_train.shuffle(512)
ds_test = raw_test

print('Dataset:', used_dataset, 'num_classes:', num_classes)
print('Train examples (dataset total):', ds_info.splits['train'].num_examples)
print('Using train split:', train_split, '| test split:', test_split)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/speech_commands/incomplete.7ZO030_0.0.3/speech_commands-train.tfrecord*...…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/speech_commands/incomplete.7ZO030_0.0.3/speech_commands-validation.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/speech_commands/incomplete.7ZO030_0.0.3/speech_commands-test.tfrecord*...:…

Dataset speech_commands downloaded and prepared to /root/tensorflow_datasets/speech_commands/0.0.3. Subsequent calls will reuse this data.
Dataset: speech_commands num_classes: 12
Train examples (dataset total): 85511
Using train split: train[:1%] | test split: test[:1%]


## Preprocessing: waveform → mel spectrogram

In [10]:
def audio_to_mel(waveform, sample_rate=16000, n_mels=64):
    if isinstance(waveform, tf.Tensor):
        waveform = waveform.numpy()
    waveform = waveform.astype('float32')
    mel = librosa.feature.melspectrogram(y=waveform, sr=sample_rate, n_fft=512, hop_length=256, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db.astype('float32')

# Prepare tf.data pipeline (map + batching)
def prepare_dataset(ds, batch_size=8):
    def _map(audio, label):
        audio = tf.cast(audio, tf.float32)
        mel = tf.numpy_function(audio_to_mel, [audio], tf.float32)
        mel.set_shape([64, None])
        mel = tf.expand_dims(mel, -1)
        mel = tf.image.resize(mel, [64, 64])
        return mel, label
    return ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE).cache()

train_ds = prepare_dataset(ds_train, batch_size=8)
test_ds = prepare_dataset(ds_test, batch_size=8)

for x, y in train_ds.take(1):
    print('batch x shape', x.shape, 'y shape', y.shape)

batch x shape (8, 64, 64, 1) y shape (8,)


## Small CNN model

In [11]:
# Use detected num_classes from the dataset info
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 1)),
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
print('Classes:', label_names)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       401,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 407,052 (1.55 MB)

 Trainable params: 407,052 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

Classes: ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes', '_silence_', '_unknown_']


In [12]:
# Train briefly to verify the pipeline; increase epochs/splits if you need better accuracy
history = model.fit(train_ds, validation_data=test_ds, epochs=2, verbose=1)

Epoch 1/2
107/107 ━━━━━━━━━━━━━━━━━━━━ 12s 55ms/step - accuracy: 0.5471 - loss: 6.6821 - val_accuracy: 0.0612 - val_loss: 2.5135
Epoch 2/2
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6716 - loss: 1.7684 - val_accuracy: 0.0612 - val_loss: 3.1319


## Notes
- This is a toy demo: for a real experiment use larger splits and more epochs.
- You can restrict to a few target words (e.g., 'yes','no') by filtering labels to simplify training.
- For production, use data augmentation (time shift, noise) and proper evaluation metrics.